In [3]:
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. Загрузка данных с защитой от ошибок пути
train_path = '/kaggle/input/competitions/mental-health-prediction-2026/train.csv'
if not os.path.exists(train_path):
  train_path = '/kaggle/input/mental-health-prediction-2026/train.csv'

test_path = '/kaggle/input/competitions/mental-health-prediction-2026/test.csv'
if not os.path.exists(test_path):
  test_path = '/kaggle/input/mental-health-prediction-2026/test.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

test_ids = test['id']

# Удаляем id и Name
train = train.drop(columns=['Name', 'id'], errors='ignore')
test = test.drop(columns=['Name', 'id'], errors='ignore')

y = train['Depression']
X = train.drop(columns=['Depression'])

# 2. Правильная предобработка со структурными нулями
full = pd.concat([X, test], axis=0)

full = full.fillna(0)

# Кодирование бинарных колонок
full['Gender'] = full['Gender'].map({'Male': 0, 'Female': 1})
full['Have you ever had suicidal thoughts ?'] = full[
    'Have you ever had suicidal thoughts ?'
].map({'Yes': 1, 'No': 0})
full['Family History of Mental Illness'] = full[
    'Family History of Mental Illness'
].map({'Yes': 1, 'No': 0})
full['Working Professional or Student'] = full[
    'Working Professional or Student'
].map({'Working Professional': 0, 'Student': 1})

full = pd.get_dummies(full)

X = full.iloc[: len(X)].copy()
test = full.iloc[len(X) :].copy()

# 3. Проверка качества на 5-фолдовой кросс-валидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs')

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(
    f'Точность на кросс-валидации: {scores.mean():.5f} (+/- {scores.std():.5f})'
)

# 4. Обучаем модель на ВСЕХ 100% данных
model.fit(X, y)

# 5. Предсказание для теста
preds = model.predict(test)

# Формируем и сохраняем сабмит
submission = pd.DataFrame({'id': test_ids, 'Depression': preds})

submission.to_csv('submission.csv', index=False)
print('Файл submission.csv успешно создан!')

Точность на кросс-валидации: 0.98202 (+/- 0.00198)
Файл submission.csv успешно создан!
